# 01 - Fase 1: el objetivo (superficie de Al2O3 con MACE)

Calcula **una sola vez** el objetivo estatico: corindon a-Al2O3 -> cara (0001) -> energia para arrancar un O superficial (**E_break**). El solido nunca se modifica.

**Settings (panel derecho):** Accelerator = GPU **P100** (mejor que T4 en float64) · Internet = On · *(opcional)* Add-ons -> Secrets: `MP_API_KEY`; sin ella se usa una celda de corindon interna. Repo privado -> secret `GITHUB_TOKEN`.

**Guardar:** Save Version -> Save & Run All (Commit). ~10-40 min.

In [ ]:
# ==== Setup comun: codigo en /tmp (efimero), estado en /kaggle/working/alumina (se guarda) ====
import os, glob, json, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/santiagovelcasallas/Data_Science_example_01.git"
BRANCH   = "claude/affectionate-johnson-q9wtj0"   # cambia a "main" tras el merge

ROOT = Path("/kaggle/working/alumina")
CODE = Path("/tmp/repo")
PKG  = CODE / "alumina_pipeline"
for d in ["data", "models", "priors", "runs"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
os.environ["ALUMINA_ROOT"] = str(ROOT)

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")   # solo si el repo es privado
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not CODE.exists():
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(CODE)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone fallo: " + r.stderr[-300:].replace(token or "\0", "***"))
print("Codigo en", PKG)

import re, sys
NOISE = re.compile(r"warn|it/s|Tip:|^\s*$|cuequivariance|torch\.load", re.I)

def sh(cmd, log, show=lambda line: not NOISE.search(line)):
    """Ejecuta un comando mostrando su progreso. Si falla, DETIENE el notebook
    mostrando las ultimas lineas del error (nada de fallos silenciosos)."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    tail = []
    with open(log, "w") as fh, subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                text=True, bufsize=1) as p:
        for line in p.stdout:
            fh.write(line)
            tail = (tail + [line])[-60:]
            if show(line):
                print(line, end="", flush=True)
    if p.returncode != 0:
        print("\n========== ERROR: ultimas lineas ==========\n" + "".join(tail))
        raise RuntimeError(f"El comando fallo (codigo {p.returncode}). Log completo: {log}")

In [ ]:
# ==== Instalacion (~3-6 min). Se fija la version de torch de Kaggle para no romper CUDA ====
torch_pin = [l for l in subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True,
                                       text=True).stdout.splitlines() if l.startswith("torch==")]
Path("/tmp/constraints.txt").write_text("\n".join(torch_pin) + "\n")
print("torch fijado:", torch_pin)
sh([sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/constraints.txt",
    "-r", PKG / "kaggle/requirements-kaggle.txt"], "/tmp/pip.log")
sh([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", PKG], "/tmp/pip_pkg.log")
sh([sys.executable, "-c", "import torch,rdkit,ase,botorch,chemprop,alumina_pipeline; "
    "print('torch', torch.__version__, '| chemprop', chemprop.__version__, '| GPU:', torch.cuda.is_available())"],
   "/tmp/check.log")
CHEMPROP = shutil.which("chemprop") or str(Path(sys.executable).parent / "chemprop")

In [ ]:
MACE_MODEL = "medium"     # small | medium | large
MACE_DTYPE = "float64"    # float64 (preciso, P100) | float32 (rapido, para probar)
SUPERCELL  = [2, 2, 1]    # [1,1,1] = prueba rapida

mp_key = secret("MP_API_KEY")
if mp_key: os.environ["MP_API_KEY"] = mp_key
print("Materials Project:", "si (mp-1143)" if mp_key else "no -> estructura interna")

import yaml
yaml.safe_dump({
    "run": {"device": "cuda", "name": "kaggle_phase1"},
    "phase1": {"calculator": "mace", "mace_model": MACE_MODEL, "mace_dtype": MACE_DTYPE,
               "supercell": SUPERCELL, "structure_source": "materials_project"},
}, open(ROOT / "overlay_phase1.yaml", "w"))

In [ ]:
sh([sys.executable, "-m", "alumina_pipeline.orchestrator", "--config", PKG / "configs/pipeline.yaml",
    "--overlay", ROOT / "overlay_phase1.yaml", "--phase1-only", "--force-phase1"], ROOT / "phase1.log")

In [ ]:
t = json.loads((ROOT / "data/phase1_target.json").read_text())
for k in ["source","calculator","n_atoms_slab","surface_energy","e_break_o","e_break_o_kj_mol"]:
    print(f"{k:>18}: {t[k]}")
if not 0.5 < t["surface_energy"] < 4.0: print("[!] gamma fuera de ~1-3 J/m2")
if not 3.0 < t["e_break_o"] < 10.0:     print("[!] E_break fuera de ~4-8 eV")

## Que guardar

| Archivo | Para que | Imprescindible |
|---|---|---|
| `alumina/data/phase1_target.json` | E_break, gamma: lo lee el nb 03 | **Si** |
| `alumina/data/al2o3_slab.extxyz` | Superficie relajada: chequeo MACE del 03 | **Si** |
| `alumina/runs/kaggle_phase1_*/` | Config efectiva (trazabilidad) | Recomendado |

En el nb 03: Add Input -> Notebook Output -> este notebook.